
# 11 — Data mapping and normalization (Cars 4 You)

**Scope:** Perform Mapping and normalization (no modeling).  
Outputs a clean dataset and a JSON processing report for traceability.

ONLY Rulebased Cleaning (no ML), Missing-Handling, Encoding-Preparation.  
No Fit, no Scalers, no Target.
We want to avoid any data leakage.

That is the reason, why we don't have to split the data into train/test here.

# Table of Contents

Take this as an example for a Table of Contents for your notebook.
We have to fix all the names and sections according to what we actually do in the notebook.

<a class="anchor" id="top"></a>

** **

1. [Importing Libraries](##1.-Importing-Libraries) <br>
    
2. [Data Access & Loading](#2.-Data-Access-&-Loading) <br>
    
3. [Type Casting](#3.-Type-Casting) <br>

4. [Duplicate Removal](#3.1-Duplicate-Removal) <br>
    
5. [Category Normalization](#3.2-Category-Normalization) <br>
   
   &emsp; 3.2.1 [Data Type Conversions](#3.2.1-Data-Type-Conversions) <br>
   
   &emsp; 3.2.2 [Encoding](#3.2.2-Encoding) <br>
   
   &emsp; 3.2.3 [Other Transformations](#3.2.3-Other-Transformations) <br>
    
   &emsp; 3.2.4 [Unique Feature-Pair Analysis](#3.2.4-Unique-Feature-Pair-Analysis) <br> 

   3.3 [Train-Test Split](#3.3-Train-Test-Split) <br>
   
   3.4 [Missing Values](#3.4-Missing-Values) <br>
    
   3.5 [Outliers](#3.5-Outliers) <br>

   3.6 [Visualisations](#3.6-Visualisations) <br><br>
   

## 1. Importing Libraries

In [1]:
import os, re, math, warnings
from pathlib import Path
from datetime import datetime
import json
import pandas as pd
import numpy as np
import re
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 160)
pd.set_option("mode.copy_on_write", True)
warnings.filterwarnings("ignore")

RANDOM_STATE = 42  # for reproducibility of any sampling


## 2. Data Access & Loading

In [2]:
# Load the data paths
data_dir = "../data/"

# Load the raw data into a pandas dataframe
train = pd.read_csv(os.path.join(data_dir, "train.csv"))
test = pd.read_csv(os.path.join(data_dir, "test.csv"))



print("Loaded shape:", train.shape)
display(train.head(3))


Loaded shape: (75973, 14)


,carID,Brand,model,year,price,transmission,mileage,fuelType,tax,mpg,engineSize,paintQuality%,previousOwners,hasDamage
0,69512,VW,Golf,2016.0,22290,Semi-Auto,28421.0,Petrol,NaN,11.417268,2.0,63.0,4.0,0.0
1,53000,Toyota,Yaris,2019.0,13790,Manual,4589.0,Petrol,145.0,47.900000,1.5,50.0,1.0,0.0
2,6366,Audi,Q2,2019.0,24990,Semi-Auto,3624.0,Petrol,145.0,40.900000,1.5,56.0,4.0,0.0


## 3. Type Casting

In [3]:

def to_int_series(s):
    return pd.to_numeric(s, errors="coerce").round().astype("Int64")
def to_float_series(s):
    return pd.to_numeric(s, errors="coerce").astype(float)

num_cols_suggest = ["price","mileage","engineSize","mpg","tax","year","previousOwners"]
for col in num_cols_suggest:
    if col in train.columns:
        if col in ["year", "previousOwners"]:
            train[col] = to_int_series(train[col])
        else:
            train[col] = to_float_series(train[col])

for c in train.select_dtypes(include="object").columns:
    train[c] = train[c].astype("string").str.strip().replace({"nan": pd.NA, "None": pd.NA, "": pd.NA})


train.head(2)

,carID,Brand,model,year,price,transmission,mileage,fuelType,tax,mpg,engineSize,paintQuality%,previousOwners,hasDamage
0,69512,VW,Golf,2016,22290.0,Semi-Auto,28421.0,Petrol,NaN,11.417268,2.0,63.0,4,0.0
1,53000,Toyota,Yaris,2019,13790.0,Manual,4589.0,Petrol,145.0,47.900000,1.5,50.0,1,0.0


In [4]:
# Types of the columns
train.dtypes

carID                      int64
Brand             string[python]
model             string[python]
year                       Int64
price                    float64
transmission      string[python]
mileage                  float64
fuelType          string[python]
tax                      float64
mpg                      float64
engineSize               float64
paintQuality%            float64
previousOwners             Int64
hasDamage                float64
dtype: object

## 4. Duplicate Removal

In [5]:
id_cols = [c for c in ["carID","id","ID"] if c in train.columns]
if id_cols:
    dup_n = train.duplicated(subset=id_cols).sum()
    train = train.drop_duplicates(subset=id_cols, keep="first")
else:
    dup_n = train.duplicated().sum()
    train = train.drop_duplicates(keep="first")

print(f"Duplicates removed: {dup_n} | New shape: {train.shape}")
test


Duplicates removed: 0 | New shape: (75973, 14)


,carID,Brand,model,year,transmission,mileage,fuelType,tax,mpg,engineSize,paintQuality%,previousOwners,hasDamage
0,89856,Hyundai,I30,2022.878006,Automatic,30700.000000,petrol,205.0,41.5,1.6,61.0,3.0,0.0
1,106581,VW,Tiguan,2017.000000,Semi-Auto,-48190.655673,Petrol,150.0,38.2,2.0,60.0,2.0,0.0
2,80886,BMW,2 Series,2016.000000,Automatic,36792.000000,Petrol,125.0,51.4,1.5,94.0,2.0,0.0
3,100174,Opel,Grandland X,2019.000000,Manual,5533.000000,Petrol,145.0,44.1,1.2,77.0,1.0,0.0
4,81376,BMW,1 Series,2019.000000,Semi-Auto,9058.000000,Diesel,150.0,51.4,2.0,45.0,4.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
32562,105775,VW,Tiguan,2017.000000,Manual,27575.000000,Petrol,145.0,46.3,1.4,94.0,1.0,0.0
32563,81363,BMW,X2,2020.000000,Automatic,1980.000000,Petrol,145.0,34.0,2.0,39.0,3.0,0.0
32564,76833,Audi,Q5,2019.000000,Semi-Auto,8297.000000,Diesel,145.0,38.2,2.0,88.0,4.0,0.0
32565,91768,Mercedes,A Class,2019.000000,Manual,-50755.210230,Petrol,145.0,28.5,1.3,81.0,1.0,0.0


## 5. Category Normalization and Mapping for the train dataset

In [6]:
#Mapping for transmission train
# Defining a normalization function for transmission
# Load the JSON mappin
with open("../mapping/transmission_mapping.json", "r", encoding="utf-8") as f:
    CANON = json.load(f)

print("\n Unique transmissions before cleaning:", train["transmission"].unique())

# Normalization function
def norm_transmission(transmission):
    if pd.isna(transmission):
        return transmission
    transmission = transmission.strip().lower()
    transmission = re.sub(r'[.,_]', ' ', transmission)
    transmission = ' '.join(transmission.split())  # Remove extra spaces
    return transmission
    
# apply normalization and canonical mapping
train['transmission'] = train['transmission'].apply(norm_transmission).map(CANON)

# unique transmission in the column transmission
unique_transmission = train['transmission'].unique()
print("\nUnique transmissions after cleaning:\n", unique_transmission)




 Unique transmissions before cleaning: <StringArray>
['Semi-Auto',    'Manual',     'anual',  'Semi-Aut', 'Automatic',    'manual',        <NA>,   'unknown',     'Manua', 'AUTOMATIC',    'MANUAL', 'semi-auto',
 'automatic',  'emi-Auto', 'SEMI-AUTO',  'SEMI-AUT',  'Automati',     'ANUAL',  'utomatic',    'unknow',  'EMI-AUTO',     'manua',      'anua',   'emi-Aut',
     'MANUA',  'emi-auto',  'UTOMATIC',   'UNKNOWN',    'nknown',  'automati',     'Other',  'semi-aut',  'AUTOMATI',   'utomati',     'nknow']
Length: 35, dtype: string

Unique transmissions after cleaning:
 ['Semi-Auto' 'Manual' 'Automatic' nan 'Unknown']


In [7]:
# Load the JSON mapping
with open("../mapping/fueltype_mapping.json", "r", encoding="utf-8") as f:
    CANON = json.load(f)

print("\nUnique fuel types before cleaning:", train["fuelType"].unique())

# Define a normalization function
def norm_fueltype(fueltype):
    if pd.isna(fueltype):
        return fueltype
    fueltype = fueltype.strip().lower()
    fueltype = re.sub(r'[.,-_]', ' ', fueltype)
    fueltype = ' '.join(fueltype.split())  # Remove extra spaces
    return fueltype


# Apply normalization and canonical mapping
train['fuelType'] = train['fuelType'].apply(norm_fueltype).map(CANON)

# Check results
print("\nUnique fuel types after cleaning:", train["fuelType"].unique())


Unique fuel types before cleaning: <StringArray>
[  'Petrol',   'Diesel',    'etrol',   'Hybrid',   'diesel',    'iesel',       <NA>,   'petrol',   'PETROL',    'Diese',    'Petro',   'DIESEL',    'petro',
   'HYBRID',     'ybri',    'Other',    'DIESE',    'Hybri',    'ETROL',    'ybrid',    'PETRO',   'hybrid',    'IESEL', 'Electric',     'ther',     'iese',
     'etro',     'ETRO',    'diese',     'Othe',    'YBRID',    'HYBRI',    'OTHER',    'other',     'IESE']
Length: 35, dtype: string

Unique fuel types after cleaning: ['Petrol' 'Diesel' 'Hybrid' nan 'Other' 'Electric']


In [8]:
# Map for brand train
# Load the mapping
with open("../mapping/brandname_mapping.json", "r", encoding="utf-8") as f:
    brandNameMapping = json.load(f)

print("\nUnique brand names before cleaning:", train["Brand"].unique())

# Normalization
def norm_brand(brand):
    if pd.isna(brand):
        return pd.NA
    brand = brand.strip().lower()
    brand = re.sub(r"[.,-_]", " ", brand)
    brand = " ".join(brand.split())
    return brand

# Cleaning (safe)
def clean_brand(col: pd.Series) -> pd.Series:
    nb = col.apply(norm_brand)
    mapped = nb.map(brandNameMapping)
    # Keep original where no mapping exists
    cleaned = col.where(mapped.isna(), mapped)
    return cleaned

# Apply cleaning
if "Brand" in train.columns:
    train["Brand"] = clean_brand(train["Brand"])

# Check result
print("\nUnique brand names before cleaning:", train["Brand"].unique())


Unique brand names before cleaning: <StringArray>
[      'VW',   'Toyota',     'Audi',     'Ford',      'BMW',    'Skoda',     'Opel', 'Mercedes',      'FOR', 'mercedes',  'Hyundai',        'w',      'ord',
       'MW',      'bmw',       <NA>,   'yundai',       'BM',    'Toyot',      'udi',      'Ope',     'AUDI',        'V',     'opel',      'pel',      'For',
       'pe',  'Mercede',     'audi', 'MERCEDES',     'OPEL',     'koda',     'FORD',   'Hyunda',        'W',      'Aud',       'vw',  'hyundai',    'skoda',
     'ford',   'TOYOTA',  'ercedes',    'oyota',   'toyota',    'SKODA',     'Skod',  'HYUNDAI',      'kod',        'v',      'for',     'SKOD',      'aud',
     'KODA',      'PEL',    'yunda',       'or',      'UDI',    'OYOTA',   'HYUNDA',       'mw',      'OPE',  'mercede',  'ERCEDES',   'ercede',    'TOYOT',
  'MERCEDE',      'ORD',       'ud',      'ope',      'AUD',   'hyunda',     'skod',    'toyot']
Length: 73, dtype: string

 <StringArray>ames before cleaning:
['Vo

In [9]:
# Load mapping JSON (can be regex-only; ALIASES may also be empty) --> This mapping was created with the help of ChatGPT and manual adjustments and checks 
with open("../mapping/modelname_mapping.json", "r", encoding="utf-8") as f:
    MM = json.load(f)

unique_models_pre = train["model"].unique()
print("\nUnique Model in 'model':\n", unique_models_pre)
print(f"\nNumber of unique models: {len(unique_models_pre)}")

ALIASES = MM.get("aliases", {})            # optional
REGEX_RULES = MM.get("regex_rules", [])    # main patterns


def norm_model(x):
    """Lowercase, collapse multiple spaces, normalize hyphens/underscores."""
    if pd.isna(x):
        return np.nan
    s = str(x).lower()
    s = re.sub(r'[-_]+', '-', s)        # replace hyphens/underscores with single hyphen
    s = re.sub(r'\s+', ' ', s).strip()  # collapse multiple spaces
    return s


def apply_regex_rules(norm_key: str):
    """Apply sequential regex rules until first successful match."""
    out = norm_key
    matched = False
    for rule in REGEX_RULES:
        pat = rule.get("pattern")
        rep = rule.get("replace")
        new = re.sub(pat, rep, out)
        if new != out:
            matched = True
        out = new
    return out, matched


def post_canon_model(model):
    """Final canonical cleanup — uppercase Mercedes classes and proper multi-word capitalization."""
    if pd.isna(model): 
        return model

    t = str(model)

    # Mercedes Classes clean up
    t = re.sub(r'([A-Za-z0-9]+)\s*-?\s*class$', lambda m: m.group(1).upper() + "-Class", t, flags=re.I)

    # BMW i / ix families
    t = re.sub(r'^i(\d+)$', lambda m: "i" + m.group(1), t, flags=re.I)
    t = re.sub(r'^ix(\d+)$', lambda m: "ix" + m.group(1), t, flags=re.I)

    # Audi / VW / Ford / Skoda special cases
    special_upper = {
        "tt": "TT", "r8": "R8", "s-max": "S-MAX", "c-max": "C-MAX"
    }
    t_lower = t.lower()
    if t_lower in special_upper:
        return special_upper[t_lower]

    # Title-case remaining multi-word models
    # But preserve i30, i8, i40 etc. (lowercase 'i')
    def title_except_i(word):
        if re.match(r'^i\d+$', word, flags=re.I):
            return word.lower()
        return word.capitalize()
    
    t = " ".join(title_except_i(w) for w in t.split())

    return t

def clean_model(series: pd.Series) -> pd.Series:
    original = series.astype("string")

    # Normalize for mapping: lowercase, remove spaces/hyphens/underscores
    normed = original.apply(norm_model)

    # Alias mapping (optional)
    alias_mapped = normed.map(ALIASES) if ALIASES else pd.Series(pd.NA, index=series.index)
    out = original.where(alias_mapped.isna(), alias_mapped)  # keep original if alias not mapped

    # Apply regex rules where alias didn't apply
    need = out.isna() | (out == original)
    if need.any():
        def _rx(key):
            if isinstance(key, str):
                # Apply all regex rules sequentially
                for rule in REGEX_RULES:
                    pat = rule.get("pattern")
                    rep = rule.get("replace")
                    new = re.sub(pat, rep, key)
                    if new != key:
                        key = new  # update key if rule matched
                return key
            return pd.NA

        # Apply regex to normalized values and immediately canonicalize
        rx_res = normed[need].apply(_rx).apply(post_canon_model)
        out.loc[need] = out.loc[need].where(rx_res.isna(), rx_res)

    # Final canonical formatting (ensures capitalization and special cases)
    out = out.apply(post_canon_model)

    return out

# Apply cleaning
if "model" in train.columns:
    train["model"] = clean_model(train["model"])

# Checks
unique_models = train["model"].unique()
print("\nUnique Model in 'model':\n", unique_models)
print(f"\nNumber of unique models: {len(unique_models)}")



Unique Model in 'model':
 <StringArray>
[         'Golf',         'Yaris',            'Q2',        'FIESTA',      '2 Series',      '3 Series',            'A3',       'Octavia',        'Passat',
         'Focus',
 ...
      '8 SERIES',          'Vers',      'Terracan', 'ZAFIRA TOURER',            'm3',           'AYG',          's-ma',            'm4',        'arteon',
     'glb class']
Length: 547, dtype: string

Number of unique models: 547

Unique Model in 'model':
 ['Golf' 'Yaris' 'Q2' 'Fiesta' '2 Series' '3 Series' 'A3' 'Octavia'
 'Passat' 'Focus' 'Insignia' 'A-class' 'Q3' 'Fabia' 'Ka+' 'Glc-class'
 'i30' 'C-class' 'Polo' 'E-class' 'Q5' 'Up' 'C-hr' 'Mokka X' 'Corsa'
 'Astra' 'TT' '5 Series' 'Aygo' '4 Series' 'Slk' 'Viva' 'T-roc' 'Ecosport'
 'Tucson' 'Ecospor' <NA> 'X-class' 'Cl-class' 'Ix20' 'i20' 'Rapid' 'A1'
 'Auris' 'Sharan' 'Adam' 'X3' 'A8' 'Gls-class' 'B-max' 'A4' 'Kona' 'i10'
 'Mokka' 'S-MAX' 'X2' 'Crossland X' 'Tiguan' 'A5' 'Gle-class' 'Zafira'
 'Ioniq' 'A6' 'Mondeo' 'Yeti 

In [10]:
unique_models = train["model"].dropna().unique().tolist()
print("\nUnique Model in 'model':\n", unique_models)
print(f"\nNumber of unique models: {len(unique_models)}")



Unique Model in 'model':
 ['Golf', 'Yaris', 'Q2', 'Fiesta', '2 Series', '3 Series', 'A3', 'Octavia', 'Passat', 'Focus', 'Insignia', 'A-class', 'Q3', 'Fabia', 'Ka+', 'Glc-class', 'i30', 'C-class', 'Polo', 'E-class', 'Q5', 'Up', 'C-hr', 'Mokka X', 'Corsa', 'Astra', 'TT', '5 Series', 'Aygo', '4 Series', 'Slk', 'Viva', 'T-roc', 'Ecosport', 'Tucson', 'Ecospor', 'X-class', 'Cl-class', 'Ix20', 'i20', 'Rapid', 'A1', 'Auris', 'Sharan', 'Adam', 'X3', 'A8', 'Gls-class', 'B-max', 'A4', 'Kona', 'i10', 'Mokka', 'S-MAX', 'X2', 'Crossland X', 'Tiguan', 'A5', 'Gle-class', 'Zafira', 'Ioniq', 'A6', 'Mondeo', 'Yeti Outdoor', 'X1', 'Scala', 'S-class', '1 Series', 'Kamiq', 'Kuga', 'Tourneo Connect', 'Q7', 'Gla-class', 'Arteon', 'Sl-class', 'Santa Fe', 'Grandland X', 'i800', 'Rav4', 'Touran', 'Citigo', 'Roomster', 'Prius', 'Corolla', 'B-class', 'Q', 'Kodiaq', 'V-class', 'Caddy Maxi Life', 'Superb', 'Astr', 'Getz', 'Combo Life', 'Beetle', 'Galaxy', 'M3', 'Gtc', 'X4', 'Ix35', 'Grand Tourneo Connect', 'Shara',

## 6. Category Normalization and Mapping for the test dataset

In [11]:
# Mapping for transmission test
# We'll use the normalization functions already defined on the train dataset category mapping
# Load the JSON mappin
with open("../mapping/transmission_mapping.json", "r", encoding="utf-8") as f:
    CANON = json.load(f)

print("Unique transmissions before cleaning:", test["transmission"].unique())

# apply normalization and canonical mapping
test['transmission'] = test['transmission'].apply(norm_transmission).map(CANON)

# unique transmission in the column transmission
unique_transmission = test['transmission'].unique()
print("\nUnique transmissions after cleaning:\n", unique_transmission)

Unique transmissions before cleaning: ['Automatic' 'Semi-Auto' 'Manual' 'unknow' 'Manua' 'automatic' nan
 'semi-auto' 'MANUAL' 'Semi-Aut' 'unknown' 'emi-Auto' 'utomatic'
 'SEMI-AUTO' 'anual' 'Automati' 'manual' 'AUTOMATIC' ' Manual ' ' Manual'
 'UNKNOWN' 'anua' 'AUTOMATI' 'nknown' 'MANUA' 'Other' ' MANUAL ' 'manual '
 'manua' 'UTOMATIC' 'automati' 'utomati' 'ANUAL' 'emi-auto' 'EMI-AUTO'
 'SEMI-AUT' 'Manual ' ' manual ' 'emi-Aut']

Unique transmissions after cleaning:
 ['Automatic' 'Semi-Auto' 'Manual' 'Unknown' nan]


In [12]:
# Map for 
# Load the JSON mapping
with open("../mapping/fueltype_mapping.json", "r", encoding="utf-8") as f:
    CANON = json.load(f)

print("\nUnique fuel types before cleanning", test["fuelType"].unique())

# Apply normalization and canonical mapping
test['fuelType'] = test['fuelType'].apply(norm_fueltype).map(CANON)

# Check results
print("\nUnique fuel types after cleaning:", test["fuelType"].unique())


Unique fuel types before cleanning ['petrol' 'Petrol' 'Diesel' 'Diese' 'Hybrid' 'iesel' 'Petro' 'hybrid'
 'etrol' 'DIESEL' nan 'PETROL' 'diesel' 'Other' 'iese' 'diese' 'etro'
 'petro' 'HYBRID' 'Hybri' 'ther' 'ETROL' 'ybrid' 'IESEL' 'DIESE' 'PETRO'
 'hybri' 'other' 'Othe' 'Electric']

Unique fuel types after cleaning: ['Petrol' 'Diesel' 'Hybrid' nan 'Other' 'Electric']


In [13]:
# Map for brand test
# Load the mapping
with open("../mapping/brandname_mapping.json", "r", encoding="utf-8") as f:
    brandNameMapping = json.load(f)
print("\nUnique brand names before cleaning:", test["Brand"].unique())

# Apply cleaning
if "Brand" in test.columns:
    test["Brand"] = clean_brand(test["Brand"])

# Check result
print("\nUnique brand names after cleaning:", test["Brand"].unique())


Unique brand names before cleaning: ['Hyundai' 'VW' 'BMW' 'Opel' 'Ford' 'Mercedes' 'Skoda' 'Toyot' 'Toyota'
 'Audi' nan 'For' 'Ope' 'toyota' 'vw' 'hyundai' 'MW' 'SKODA' 'ord' 'udi'
 'bmw' 'V' 'BM' 'HYUNDAI' 'OPEL' 'mercedes' 'audi' 'Mercede' 'pel' 'opel'
 'FORD' 'yundai' 'ford' 'Aud' 'oyota' 'MERCEDES' 'ercedes' 'AUDI' 'koda'
 'Hyunda' 'W' 'skoda' 'Skod' 'ercede' 'TOYOTA' 'ERCEDES' 'kod' 'ORD' 'v'
 'ud' 'M' 'FOR' 'for' 'MERCEDE' 'YUNDAI' 'PEL' 'ope' 'or' 'TOYOT' 'hyunda'
 'oyot' 'UDI' 'mw' 'pe' 'bm']

Unique brand names after cleaning: ['Hyundai' 'Volkswagen' 'BMW' 'Opel' 'Ford' 'Mercedes-Benz' 'Škoda'
 'Toyota' 'Audi' nan]


In [14]:
# Load mapping JSON (can be regex-only; ALIASES may also be empty) --> This mapping was created with the help of ChatGPT and manual adjustments and checks 
with open("../mapping/modelname_mapping.json", "r", encoding="utf-8") as f:
    MM = json.load(f)

unique_models_pre = test["model"].unique()
print("\nUnique Model before cleaning':\n", unique_models_pre)
print(f"\nNumber of unique models: {len(unique_models_pre)}")

ALIASES = MM.get("aliases", {})            # optional
REGEX_RULES = MM.get("regex_rules", [])    # main patterns

# Apply cleaning
if "model" in test.columns:
    test["model"] = clean_model(train["model"])

# Checks
unique_models = test["model"].unique()
print("\nUnique Model after cleaning':\n", unique_models)
print(f"\nNumber of unique models: {len(unique_models)}")


Unique Model before cleaning':
 [' I30' ' Tiguan' ' 2 Series' ' Grandland X' '1 Series' ' Fiesta' ' X1'
 ' B Class' ' Focus' ' Superb' ' 5 Series' ' C Class' ' Up' ' Aygo' 'Golf'
 ' M CLAS' ' Land Cruiser' ' TT' ' Adam' ' Zafira' ' E Class' ' Golf'
 ' 3 Series' ' IX20' ' A4' ' Yaris' ' Passat' ' I10' ' Mokka X'
 ' EcoSport' ' 1 Series' ' 4 Series' ' A7' ' Corsa' ' Kuga' ' Grand C-MAX'
 ' Q2' ' M4' ' A Class' ' RAV4' ' Fabia' ' Insignia' ' A1' ' X6' ' Meriva'
 ' Caravelle' ' Octavia' ' Auris' ' X-CLASS' ' FOCUS' ' Astra' ' V Class'
 ' Polo' ' Karoq' ' Shuttle' ' Mokka' ' Q5' ' Tucson' ' A3' ' SL CLASS'
 ' Corolla' ' a class' ' Ka+' ' X3' ' I40' ' I20' ' Kamiq' nan ' IX35'
 ' Crossland X' ' Q3' ' Viva' ' GLA Class' ' Astr' ' CLS Class' ' KA'
 ' FABIA' ' a3' 'Polo' ' Focu' ' Galaxy' ' X2' ' Kodiaq' ' GLC Class'
 ' Vivaro' ' Mondeo' ' Touran' ' CL Class' ' CORSA' 'Aygo' ' X5' ' FIESTA'
 'Auris' ' mokka x' ' Verso' ' Touareg' ' T-Roc' ' fiesta' ' Fiest' ' Q'
 ' A5' ' S Class' ' C Clas' ' S

### We skip the Canonicalization of Models for now. to be revisited later.


## 8. Process errors found in data exploration

In [15]:
# we cannot just round them, because they wouldnt match the other values, so we will set them na
train.loc[train['year'] % 1 != 0, 'year'] = np.nan
# its obviously no sign error, so the values will be set to na for now
train.loc[train['year'] < 0, 'mileage'] = np.nan

In [16]:
# The negative values can't really be explained, so we will set them to na
train.loc[train['tax'] < 0, 'tax'] = np.nan

In [17]:
# the values just look wrong, so we will set all unusual values to na
train.loc[((train['engineSize'] > 6.2) | (train['engineSize'] < 0.8)), 'engineSize'] = np.nan

In [18]:
# its obviously no sign error, so the values will be set to na for now
train.loc[train['mileage'] < 0, 'mileage'] = np.nan

In [19]:
# Some mpg values could be unrealistic as well
unusual_mpg_mask = ((train['mpg'] > 150) & (train["fuelType"] == "Electric")) | \
       ((train['mpg'] < 70) & (train["fuelType"] == "Electric")) | \
       ((train['mpg'] > 100) & (train["fuelType"] == "Hybrid")) | \
       ((train['mpg'] < 35) & (train["fuelType"] == "Hybrid")) | \
       ((train['mpg'] > 80) & (train["fuelType"] != "Hybrid") & (train["fuelType"] != "Electric")) | \
       ((train['mpg'] < 8) & (train["fuelType"] != "Hybrid")& (train["fuelType"] != "Electric"))
train.loc[unusual_mpg_mask, 'mpg'] = np.nan

## 10. Save Processed Data

In [20]:
# save the processed dataframe to data/processed_data
PROCESSED_CSV = os.path.join(data_dir, "processed_data/11_processed_train_data.csv")
print("Saving processed file to:", PROCESSED_CSV)
train.to_csv(PROCESSED_CSV, index=False)
print("✅ Saved. Rows with any remaining NaNs:", int(train.isna().any(axis=1).sum()))

Saving processed file to: ../data/processed_data/11_processed_train_data.csv
✅ Saved. Rows with any remaining NaNs: 23509
